# Level 8 on Colab: fine-tuning with training queries written by an LLM

In Level 6, fine-tuning did not help because the T5 training queries were questions like
"what is a hose nozzle". Here an instruction LLM (Qwen2.5-3B) writes 3 shopping searches per product:
the kind of product, a need it solves, and a feature. Then gte-small, the best model so far,
is trained on them and compared with gte-small without training.

Before you start: Runtime → Change runtime type → T4 GPU. Then run the cells from top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the project (code + labels)

In [ ]:
%cd /content
!rm -rf Semantic_Search   # start fresh, so a re-run gets the newest code and labels
!git clone -q https://github.com/leilaalizadeh/Semantic_Search.git
%cd Semantic_Search

## 3. Install the project and download the products
The same 5,000 Amazon products as on the laptop, so the labels match.

In [ ]:
!pip install -q -e .
!python -m semantic_search.levels.level8.prepare_data

## 4. The LLM writes the training queries (about 30 to 60 minutes)
If the GPU runs out of memory, add `--batch-size 16`.

In [ ]:
!python -m semantic_search.levels.level8.generate_llm_queries
!head -n 12 data/train/llm_queries.jsonl

## 5. Fine-tune gte-small on them (a few minutes)
On a GPU we can use bigger batches (64) and longer texts (256 tokens) than on the laptop.

In [ ]:
!python -m semantic_search.levels.level8.train --data llm --batch-size 64 --max-length 256

## 6. Compare with gte-small without training
Look at the `unlabeled` column: products only the new model finds were never labeled, so its score
here is a first look. The fair score comes after labeling them on the laptop.

In [ ]:
!python -m semantic_search.levels.level8.compare_models --models gte-small,gte-small-llm

## 7. Download the model and the training queries
On the laptop: unzip the model into `models/` and put the queries into `data/train/`.

In [ ]:
!cd models && zip -qr gte-small-llm.zip gte-small-llm
from google.colab import files
files.download("models/gte-small-llm.zip")
files.download("data/train/llm_queries.jsonl")